# DevForge Track 1: Can It Play DOOM? — Colab GPU Training Runner

**Team:** Prit (ML Lead), Surbhi (Curriculum), Vaibhav (Infra & Training), Dia (Evaluation)

### Overnight Run Schedule (IST):
- **22:00 Thu:** Launch Stage 1–5 Curriculum Run on Google Colab GPU (T4/A100)
- **00:00–02:30 Fri:** Active Watch: Vaibhav (check loss, step progress, Drive backups)
- **02:30–05:00 Fri:** Active Watch: Surbhi
- **05:00–07:30 Fri:** Active Watch: Dia
- **07:30–08:00 Fri:** Best Checkpoint Selection & Evaluation


In [ ]:
# 1. GPU Check & Environment Verification
!nvidia-smi
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")


In [ ]:
# 2. Clone Public Repository & Pull Git LFS Weights
!apt-get install -y git-lfs
!git lfs install
!rm -rf CanItDoom
!git clone https://github.com/imagine1phoenix/CanItDoom.git
%cd CanItDoom
!git lfs pull
!ls -lh submission/weights/model.pt


In [ ]:
# 3. Install Dependencies
!pip install --upgrade pip
!pip install uv
!uv pip install --system -e env/


In [ ]:
# 4. Prepare Local Checkpoint Directory (No Google Drive mount per CONTRIBUTING.md)
import os
os.makedirs("checkpoints", exist_ok=True)
print("Local checkpoints directory ready:", os.path.abspath("checkpoints"))


In [ ]:
# 5. Pre-Training Rule & Baseline Validation Check
!python3 tools/check_rules.py
!OMP_NUM_THREADS=1 python validate.py


In [ ]:
# 6. Launch Parallel Recurrent PPO Curriculum Training (Stages 1-5)
# Saves atomic checkpoints every 15 min (download checkpoints/ckpt_latest.pt periodically)
!python train/run.py --stages 1,2,3,4,5 --steps-per-stage 1000000 --envs 8 --device cuda --save-interval-sec 900


In [ ]:
# 7. Evaluate Checkpoint on 6 Held-Out Evaluation Maps
!python train/evaluate.py --checkpoint checkpoints/ckpt_latest.pt --output eval_results.csv --device cuda
!cat eval_results.csv


In [ ]:
# 8. Synchronize Best Checkpoint to submission/weights/model.pt & Final Verification
!cp checkpoints/ckpt_latest.pt submission/weights/model.pt
!python3 tools/check_rules.py
!python validate.py


In [ ]:
# 9. Visual Inspection & Gameplay Video Playback
# Renders the 3-panel HUD (DOOM camera, 90-deg ray wireframe, polar radar) and displays inline video
!python train/visualize.py --map maps/stage1/s1_seed000.wad:MAP01 --checkpoint submission/weights/model.pt --headless --record gameplay.mp4 --max-steps 300
from IPython.display import HTML
from base64 import b64encode
mp4 = open('gameplay.mp4', 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
HTML(f'''<video width=900 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>''')
